# 08 · MCP: One Protocol for Tools

**Where we are in the stack:** the **capability table goes on the wire**. Same loop, but the
tools now live in a *separate process*, discovered and called over the
**Model Context Protocol (MCP)**.

So far every tool was a Python function in the same kernel as the loop. That is N×M glue:
every agent re-implements every integration. Networking solved this class of problem long
ago - you don't write a custom driver per vendor, you speak **SNMP/gNMI to a standard MIB**.
MCP is that move for tools: a server *advertises* its tools (name + JSON schema), any client
*discovers* and calls them over a standard transport (stdio here; HTTP in production).

```
agent loop  --MCP client-->  [stdio]  --MCP server-->  calculate_subnet / get_interface_status
```

The tools themselves are the exact ones from notebook 02. What changes is *where they run*
and *how they are found*.

> Needs the `mcp` package (installed below) and a tool-capable model (see notebook 02).

In [ ]:
# --- Provider config: works with OpenAI, OpenRouter, or a local OpenAI-compatible server ---
import os
from openai import OpenAI

# Load settings from a .env file if present (falls back to existing env vars).
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    import os
    if os.path.exists(".env"):
        for _line in open(".env"):
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())


# Pick ONE setup by exporting these env vars before launching Jupyter.
#
#   OpenAI:     OPENAI_BASE_URL=https://api.openai.com/v1   MODEL=gpt-4o-mini
#   OpenRouter: OPENAI_BASE_URL=https://openrouter.ai/api/v1 MODEL=openai/gpt-4o-mini
#   Local:      OPENAI_BASE_URL=http://localhost:11434/v1    MODEL=qwen2.5:7b   (Ollama)
#               (use 'qwen2.5' / 'llama3.1' etc. - a 1B model is great for chat but
#                usually too weak to drive tool-calling reliably.)

BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
API_KEY  = os.environ.get("OPENAI_API_KEY", "set-me")   # any non-empty string for local servers
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

# Behind a TLS-intercepting firewall/proxy, HTTPS cert verification can fail.
# Set VERIFY_SSL=false in .env to skip it: we hand the OpenAI SDK a custom
# httpx client with verification turned off. Leave it true everywhere else.
import httpx
VERIFY_SSL = os.environ.get("VERIFY_SSL", "true").strip().lower() not in ("false", "0", "no")
http_client = httpx.Client(verify=VERIFY_SSL)
if not VERIFY_SSL:
    import warnings
    warnings.filterwarnings("ignore")
    print("\u26a0\ufe0f  SSL verification DISABLED (VERIFY_SSL=false) \u2014 use only on a trusted network")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, http_client=http_client)
print("endpoint:", BASE_URL, "| model:", MODEL)

In [ ]:
# Install the MCP SDK (run once).
%pip install -q "mcp>=1.0"

## 1. Write the MCP server (a separate process)

`%%writefile` saves this cell as a standalone script - **it is not run here**. The client
will launch it as a subprocess and talk to it over stdio. `FastMCP` turns each decorated
function into an advertised tool, generating the JSON schema from the signature + docstring
(notebook 03's `@tool` did the same trick inside LangChain; MCP does it *across processes*).

Note what we did **not** write: any agent code. The server knows nothing about loops or
models. It only exports capabilities - like a device exposing a MIB, not caring who polls it.

In [ ]:
%%writefile mcp_network_server.py
"""A minimal MCP server exposing notebook 02's two network tools over stdio.

Run by the MCP client as a subprocess - not imported by the notebook.
"""
import hashlib
import ipaddress

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("network-ops")


@mcp.tool()
def calculate_subnet(cidr: str) -> dict:
    """Compute network, broadcast, netmask and usable host count for an IPv4/IPv6 CIDR."""
    net = ipaddress.ip_network(cidr, strict=False)
    if net.version == 4:
        usable = net.num_addresses - 2 if net.prefixlen <= 30 else net.num_addresses
    else:
        usable = net.num_addresses
    return {
        "network": str(net.network_address),
        "broadcast": str(net.broadcast_address) if net.version == 4 else "n/a",
        "netmask": str(net.netmask),
        "prefix_length": net.prefixlen,
        "total_addresses": net.num_addresses,
        "usable_hosts": usable,
    }


@mcp.tool()
def get_interface_status(device: str, interface: str) -> dict:
    """Operational status and error counters for an interface on a device (MOCK telemetry)."""
    h = int(hashlib.md5(f"{device}{interface}".encode()).hexdigest(), 16)
    up = (h % 5 != 0)  # ~80 percent up, deterministic so demos are repeatable
    return {
        "device": device, "interface": interface,
        "admin_status": "up",
        "oper_status": "up" if up else "down",
        "speed": "10Gbps", "mtu": 1500,
        "input_errors": h % 7, "output_errors": h % 3, "crc_errors": h % 4,
    }


if __name__ == "__main__":
    mcp.run(transport="stdio")   # serve tools over stdin/stdout

## 2. Discover the tools from a client

The client spawns the server and asks *"what can you do?"* (`list_tools`). Nothing below is
hard-coded to subnets or interfaces - point `StdioServerParameters` at **any** MCP server
(GitHub, Postgres, your NMS...) and the same code lists its tools.

Jupyter runs an asyncio event loop already, so we can `await` at the top level of a cell.

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER = StdioServerParameters(command=sys.executable, args=["mcp_network_server.py"])

async def discover():
    async with stdio_client(SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return (await session.list_tools()).tools

for t in await discover():
    print(f"- {t.name}: {t.description}")
    print(f"    schema: {t.inputSchema.get('properties', {})}")

## 3. Adapt discovered tools to the model's format

The model still expects the OpenAI `tools=` JSON shape from notebook 02. The adapter below is
mechanical: MCP's advertised `(name, description, inputSchema)` *is* that schema, just in a
different envelope. This ~10-line translation is exactly what agent frameworks and IDEs ship
built-in - now you have seen the whole trick.

In [ ]:
import json

def to_openai_tools(mcp_tools):
    """MCP tool advertisements -> the OpenAI 'tools=' schema the loop already uses."""
    return [
        {"type": "function", "function": {
            "name": t.name,
            "description": t.description or "",
            "parameters": t.inputSchema,
        }}
        for t in mcp_tools
    ]

print(json.dumps(to_openai_tools(await discover()), indent=2)[:400], "...")

## 4. The agent loop over MCP (same FSM, remote capability table)

One difference from notebook 02: instead of `TOOL_REGISTRY[name](**args)` calling a local
function, the loop awaits `session.call_tool(name, args)` - the request crosses a process
boundary, the result comes back over stdio. The state machine is otherwise unchanged.

In [ ]:
import json

SYSTEM = "You are a network operations assistant. Use tools when they help. Be concise."

async def run_agent_mcp(user_query, max_iterations=5, temperature=0):
    async with stdio_client(SERVER) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = to_openai_tools((await session.list_tools()).tools)  # discovered, not hard-coded

            messages = [
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": user_query},
            ]
            print("USER:", user_query); print("=" * 64)

            for step in range(1, max_iterations + 1):
                resp = client.chat.completions.create(
                    model=MODEL, messages=messages, tools=tools, temperature=temperature,
                )
                msg = resp.choices[0].message

                # TERMINATION: model asked for no tool -> it has a final answer.
                if not msg.tool_calls:
                    print(f"[step {step}] FINAL ANSWER\n{msg.content}")
                    return msg.content

                messages.append({
                    "role": "assistant",
                    "content": msg.content or "",
                    "tool_calls": [
                        {"id": tc.id, "type": "function",
                         "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                        for tc in msg.tool_calls
                    ],
                })

                for tc in msg.tool_calls:
                    name = tc.function.name
                    args = json.loads(tc.function.arguments)
                    print(f"[step {step}] MCP CALL  -> {name}({args})")
                    result = await session.call_tool(name, args)   # <-- over the wire
                    text = "\n".join(c.text for c in result.content if hasattr(c, "text"))
                    print(f"[step {step}] MCP RESULT <- {text[:200]}")
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": text[:4000],
                    })

            return "Stopped: hit max_iterations (TTL expired)."

## 5. Run it - the notebook-02 questions, now answered over MCP

In [ ]:
_ = await run_agent_mcp(
    "Is interface ethernet1/0/1 on leaf-01 operational, "
    "and how many usable hosts are in 10.20.0.0/22?"
)

## Recap

The loop never changed; the **capability table became a network service**:

- The **server** exports plain functions; `FastMCP` derives their schemas. It knows nothing
  about agents or models - a MIB, not a poller.
- The **client** *discovers* tools at runtime and a ~10-line adapter turns the advertisement
  into the `tools=` schema the model already understands.
- The FSM's only change: the execute step awaits a protocol call instead of a local function.

Why this matters: write the integration **once**, and every MCP client can use it - your
hand-rolled loop, an IDE, Claude Desktop. And the safety story is unchanged: this server is
still read-only/mocked; a real one gates its own mutations server-side, where the client
cannot bypass them.

Here we used stdio (client spawns the server). In production the same protocol runs over
HTTP to shared, remote servers - same discovery, same calls.

**Next:** making the model's *output* machine-reliable, not just its tool calls. ->
`09_structured_outputs.ipynb`